# Notebook 03: Reward Modeling from Scratch

**Series**: Frontier ML Interview Prep -- RLHF Deep Dive

**Goal**: Build a reward model end-to-end: understand Bradley-Terry, implement the loss, train on real preference data, diagnose failure modes.

---

## 1. Self-Quiz (Active Recall)

Before reading any material, answer these from memory. Write your answers in the empty cell below, then compare with the content that follows.

1. **What is a reward model?** What role does it play in RLHF?
2. **Why not use raw human ratings directly** (e.g., Likert scores) as the reward signal?
3. **What is the Bradley-Terry model?** Write the formula from memory.
4. **What data format** does reward model training require? Give a concrete example.
5. **Name two failure modes** of reward models.
6. **What happens** when you over-optimize against a reward model?

*Your answers here (double-click to edit):*

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---
## 2. Setup

In [ ]:
# Install dependencies (Colab-compatible)
!pip install -q torch transformers datasets accelerate matplotlib numpy tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 3. Reward Model Intuition

### What does a reward model do?

A reward model (RM) is a **learned scoring function** $r_\theta(x, y) \to \mathbb{R}$ that takes a prompt $x$ and a response $y$ and outputs a **scalar reward**. Higher reward = better response.

### Why not use human ratings directly?

| Approach | Problem |
|----------|--------|
| Raw Likert scores (1-5) | Noisy, miscalibrated across raters, expensive per RL step |
| Human-in-the-loop each RL step | Far too slow -- PPO needs thousands of reward queries per batch |
| **Pairwise preferences** | Cheaper, more reliable, natural for humans |

Humans are **much better at relative comparisons** than absolute scoring. "Is A better than B?" is easier and more consistent than "Rate A on a scale of 1-7."

### The preference data format

Each training example is a **triple**: `(prompt, chosen_response, rejected_response)`

```
{
    "prompt": "Explain quantum computing in simple terms.",
    "chosen": "Quantum computing uses qubits that can be 0, 1, or both at once...",
    "rejected": "Quantum computing is a type of computing that uses quantum bits..."
}
```

The RM must learn: $r_\theta(x, y_{\text{chosen}}) > r_\theta(x, y_{\text{rejected}})$

In [ ]:
# Let's see what real preference data looks like
# We'll use the Anthropic HH-RLHF dataset

example_preference = {
    "prompt": "What's the best way to learn programming?",
    "chosen": (
        "Start with a beginner-friendly language like Python. Work through "
        "interactive tutorials, build small projects that interest you, and "
        "gradually increase complexity. Practice consistently -- even 30 minutes "
        "daily is better than occasional long sessions."
    ),
    "rejected": (
        "Just read a textbook on computer science and memorize the syntax "
        "of a programming language. Once you know the syntax you can program anything."
    ),
}

print("=== Preference Data Example ===")
for key, val in example_preference.items():
    print(f"\n[{key.upper()}]:\n{val}")

---
## 4. Bradley-Terry Model Derivation

### Step-by-step derivation

We want to model the probability that response $y_w$ (winner) is preferred over $y_l$ (loser), given prompt $x$.

**Step 1: Assign latent strength.** Each response has a latent "quality" score $r_\theta(x, y)$.

**Step 2: Bradley-Terry assumption.** The probability of preferring $y_w$ over $y_l$ depends only on the *difference* in their scores:

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)$$

where $\sigma(z) = \frac{1}{1+e^{-z}}$ is the sigmoid.

**Intuition**: If $r(x, y_w) \gg r(x, y_l)$, then $\sigma(\text{large positive}) \approx 1$. The model is very confident in the preference.

**Step 3: Maximum likelihood.** Given dataset $\mathcal{D} = \{(x^{(i)}, y_w^{(i)}, y_l^{(i)})\}_{i=1}^N$:

$$\mathcal{L}(\theta) = -\frac{1}{N} \sum_{i=1}^{N} \log \sigma\big(r_\theta(x^{(i)}, y_w^{(i)}) - r_\theta(x^{(i)}, y_l^{(i)})\big)$$

**Key insight**: This is literally **binary cross-entropy** where:
- The "logit" is the reward difference $r_w - r_l$
- The "label" is always 1 (the chosen response is always the positive class)

### Why Bradley-Terry and not something else?

- It's the **simplest model** consistent with the axiom that preference depends on reward difference.
- It's the **logistic analog of the Thurstone Case V model** (Thurstone Case V assumes Gaussian noise; Bradley-Terry uses logistic noise).
- It gives a **proper scoring rule** -- the optimal RM recovers the true human preference distribution.
- The reward function is identified **up to a constant** -- adding $c$ to all rewards doesn't change preferences.

In [ ]:
# The Bradley-Terry loss in 5 lines of PyTorch

def bradley_terry_loss(reward_chosen: torch.Tensor, reward_rejected: torch.Tensor) -> torch.Tensor:
    """Compute Bradley-Terry pairwise preference loss.
    
    Args:
        reward_chosen: Scalar rewards for chosen responses, shape (batch_size,)
        reward_rejected: Scalar rewards for rejected responses, shape (batch_size,)
    
    Returns:
        Scalar loss (to minimize)
    """
    # reward difference: positive if RM correctly ranks chosen > rejected
    reward_diff = reward_chosen - reward_rejected
    # BT loss = -log(sigmoid(reward_diff)) = log(1 + exp(-reward_diff))
    loss = -F.logsigmoid(reward_diff).mean()
    return loss

# Sanity check: when chosen >> rejected, loss should be ~0
r_chosen = torch.tensor([5.0, 3.0, 10.0])
r_rejected = torch.tensor([1.0, -1.0, 2.0])
print(f"Loss when chosen >> rejected: {bradley_terry_loss(r_chosen, r_rejected):.4f}")

# When chosen << rejected, loss should be large
r_chosen_bad = torch.tensor([1.0, -1.0, 2.0])
r_rejected_bad = torch.tensor([5.0, 3.0, 10.0])
print(f"Loss when chosen << rejected: {bradley_terry_loss(r_chosen_bad, r_rejected_bad):.4f}")

# When equal, loss = log(2) ~ 0.693
r_equal = torch.tensor([1.0, 1.0, 1.0])
print(f"Loss when equal: {bradley_terry_loss(r_equal, r_equal):.4f}")
print(f"log(2) = {np.log(2):.4f}")

In [ ]:
# Visualize: how does the BT loss behave as a function of reward difference?

reward_diffs = torch.linspace(-6, 6, 200)
losses = -F.logsigmoid(reward_diffs)
probs = torch.sigmoid(reward_diffs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(reward_diffs.numpy(), losses.numpy(), 'b-', linewidth=2)
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Reward difference (r_chosen - r_rejected)', fontsize=11)
ax1.set_ylabel('BT Loss', fontsize=11)
ax1.set_title('Bradley-Terry Loss', fontsize=13)
ax1.grid(True, alpha=0.3)

ax2.plot(reward_diffs.numpy(), probs.numpy(), 'r-', linewidth=2)
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Reward difference (r_chosen - r_rejected)', fontsize=11)
ax2.set_ylabel('P(chosen > rejected)', fontsize=11)
ax2.set_title('Predicted preference probability', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Loss is convex and monotonically decreasing in reward_diff")
print("- At reward_diff=0, loss=log(2), probability=0.5 (50-50 coin flip)")
print("- Gradient is strongest near 0 -- drives separation between chosen/rejected")

---
## 5. Building the Reward Model

Architecture: **Pretrained LM backbone** + **scalar reward head**

```
Input tokens  -->  [Pretrained Transformer Encoder]  -->  Hidden states
                                                            |
                                                    [Last token h_T]
                                                            |
                                                    [Linear: d -> 1]
                                                            |
                                                     Scalar reward
```

**Key design decisions:**
- Which token's hidden state? **Last token** (for causal LMs) or **[CLS]/mean-pool** (for encoders)
- Remove the LM head (vocabulary projection) -- we don't need next-token prediction
- Single linear layer to scalar -- keeps it simple, RM capacity comes from the backbone

In [ ]:
class RewardModel(nn.Module):
    """Reward model: pretrained LM backbone + scalar reward head.
    
    Uses a pretrained encoder (e.g., distilbert) as the backbone.
    Removes the LM head and adds a linear projection to a scalar reward.
    """
    
    def __init__(self, model_name: str = "distilbert-base-uncased", dropout: float = 0.1):
        super().__init__()
        # Load pretrained backbone (no LM head -- just the encoder)
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # Scalar reward head
        self.reward_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),  # project to scalar
        )
    
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Forward pass: encode input, extract representation, project to scalar.
        
        Args:
            input_ids: Token IDs, shape (batch_size, seq_len)
            attention_mask: Attention mask, shape (batch_size, seq_len)
            
        Returns:
            rewards: Scalar rewards, shape (batch_size,)
        """
        # Get hidden states from backbone
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden)
        
        # Strategy: use the last non-padding token's hidden state
        # For each sequence, find the index of the last real token
        # attention_mask is 1 for real tokens, 0 for padding
        seq_lengths = attention_mask.sum(dim=1) - 1  # (batch,) -- 0-indexed
        batch_indices = torch.arange(hidden_states.size(0), device=hidden_states.device)
        last_token_hidden = hidden_states[batch_indices, seq_lengths]  # (batch, hidden)
        
        # Project to scalar reward
        rewards = self.reward_head(last_token_hidden).squeeze(-1)  # (batch,)
        return rewards


# Test the model architecture
print("Loading reward model backbone...")
reward_model = RewardModel("distilbert-base-uncased").to(device)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Quick forward pass test
test_inputs = tokenizer(
    ["This is a helpful response.", "This is a bad response."],
    return_tensors="pt", padding=True, truncation=True, max_length=128
).to(device)

with torch.no_grad():
    test_rewards = reward_model(**test_inputs)
print(f"\nTest rewards: {test_rewards.cpu().numpy()}")
print(f"Model parameters: {sum(p.numel() for p in reward_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in reward_model.parameters() if p.requires_grad):,}")

In [ ]:
# Architecture choices discussion

print("=== Reward Model Architecture Decisions ===")
print()
print("1. WHICH TOKEN to use for the reward?")
print("   - Last token (causal LMs): natural choice, has seen all previous context")
print("   - [CLS] token (BERT-style): specifically designed for sequence-level tasks")
print("   - Mean pooling: averages all token representations, more robust")
print("   - We use last-non-padding-token: works for both causal and encoder models")
print()
print("2. POOLING strategy alternatives:")
print("   - Last token: simple, standard in practice (InstructGPT)")
print("   - Mean pool: more stable gradients, but dilutes with padding")
print("   - Attention-weighted pool: learned, but adds complexity")
print()
print("3. REWARD HEAD design:")
print("   - Single linear layer (standard): backbone does the heavy lifting")
print("   - MLP head: slight improvement, risk of overfitting")
print("   - In practice, single linear + dropout is preferred")
print()
print("4. BACKBONE choice:")
print("   - Same architecture as policy (InstructGPT approach)")
print("   - Separate, often smaller model (efficiency)")
print("   - We use DistilBERT for Colab -- production uses 6B+ parameter models")

---
## 6. Training on Real Preference Data

We use the **Anthropic HH-RLHF** dataset -- real human preference data collected for training helpful and harmless assistants.

In [ ]:
# Load Anthropic HH-RLHF dataset (small subset for Colab)
print("Loading Anthropic HH-RLHF dataset...")
raw_dataset = load_dataset("Anthropic/hh-rlhf", split="train")

# Use a small subset for tractability on Colab
TRAIN_SIZE = 2000
EVAL_SIZE = 500

# Shuffle and split
raw_dataset = raw_dataset.shuffle(seed=42)
train_raw = raw_dataset.select(range(TRAIN_SIZE))
eval_raw = raw_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + EVAL_SIZE))

print(f"Train size: {len(train_raw)}")
print(f"Eval size: {len(eval_raw)}")
print(f"\nExample entry keys: {list(train_raw[0].keys())}")
print(f"\nChosen (first 200 chars): {train_raw[0]['chosen'][:200]}")
print(f"\nRejected (first 200 chars): {train_raw[0]['rejected'][:200]}")

In [ ]:
class PreferenceDataset(Dataset):
    """Dataset for reward model training on pairwise preferences.
    
    Each item returns tokenized chosen and rejected responses.
    The Anthropic HH dataset has 'chosen' and 'rejected' fields
    that include the full conversation (prompt + response).
    """
    
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize chosen response (full conversation)
        chosen_enc = self.tokenizer(
            item['chosen'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        
        # Tokenize rejected response (full conversation)
        rejected_enc = self.tokenizer(
            item['rejected'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        
        return {
            'chosen_input_ids': chosen_enc['input_ids'].squeeze(0),
            'chosen_attention_mask': chosen_enc['attention_mask'].squeeze(0),
            'rejected_input_ids': rejected_enc['input_ids'].squeeze(0),
            'rejected_attention_mask': rejected_enc['attention_mask'].squeeze(0),
        }


# Create datasets and dataloaders
train_dataset = PreferenceDataset(train_raw, tokenizer, max_length=256)
eval_dataset = PreferenceDataset(eval_raw, tokenizer, max_length=256)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Eval batches: {len(eval_loader)}")

# Inspect one batch
batch = next(iter(train_loader))
print(f"\nBatch keys: {list(batch.keys())}")
print(f"Chosen input_ids shape: {batch['chosen_input_ids'].shape}")

In [ ]:
def train_reward_model(
    model,
    train_loader,
    eval_loader,
    num_epochs=3,
    lr=1e-5,
    eval_every_steps=50,
):
    """Train the reward model using Bradley-Terry loss on preference pairs."""
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Tracking
    train_losses = []
    eval_losses = []
    eval_accuracies = []
    step_numbers = []
    global_step = 0
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            # Move to device
            chosen_ids = batch['chosen_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            
            # Forward pass: get rewards for both chosen and rejected
            reward_chosen = model(chosen_ids, chosen_mask)
            reward_rejected = model(rejected_ids, rejected_mask)
            
            # Bradley-Terry loss
            loss = bradley_terry_loss(reward_chosen, reward_rejected)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Track
            train_losses.append(loss.item())
            epoch_loss += loss.item()
            global_step += 1
            
            # Compute batch accuracy
            with torch.no_grad():
                batch_acc = (reward_chosen > reward_rejected).float().mean().item()
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'acc': f"{batch_acc:.3f}",
                'r_cho': f"{reward_chosen.mean().item():.2f}",
                'r_rej': f"{reward_rejected.mean().item():.2f}",
            })
            
            # Periodic evaluation
            if global_step % eval_every_steps == 0:
                eval_loss, eval_acc = evaluate_reward_model(model, eval_loader)
                eval_losses.append(eval_loss)
                eval_accuracies.append(eval_acc)
                step_numbers.append(global_step)
                print(f"  Step {global_step}: eval_loss={eval_loss:.4f}, eval_acc={eval_acc:.3f}")
                model.train()
        
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} avg train loss: {avg_loss:.4f}")
    
    return {
        'train_losses': train_losses,
        'eval_losses': eval_losses,
        'eval_accuracies': eval_accuracies,
        'step_numbers': step_numbers,
    }


def evaluate_reward_model(model, eval_loader):
    """Evaluate RM on held-out preference pairs."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_pairs = 0
    
    with torch.no_grad():
        for batch in eval_loader:
            chosen_ids = batch['chosen_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            
            reward_chosen = model(chosen_ids, chosen_mask)
            reward_rejected = model(rejected_ids, rejected_mask)
            
            loss = bradley_terry_loss(reward_chosen, reward_rejected)
            total_loss += loss.item() * chosen_ids.size(0)
            total_correct += (reward_chosen > reward_rejected).sum().item()
            total_pairs += chosen_ids.size(0)
    
    avg_loss = total_loss / total_pairs
    accuracy = total_correct / total_pairs
    return avg_loss, accuracy

In [ ]:
# Train the reward model
print("Training reward model on Anthropic HH-RLHF data...\n")

# Re-initialize for clean training
reward_model = RewardModel("distilbert-base-uncased").to(device)

history = train_reward_model(
    model=reward_model,
    train_loader=train_loader,
    eval_loader=eval_loader,
    num_epochs=3,
    lr=2e-5,
    eval_every_steps=25,
)

print("\nTraining complete!")

**Insider Tip:** Reward model quality is the single biggest bottleneck in RLHF. At frontier labs, significant engineering effort goes into reward model training data collection, inter-annotator agreement measurement, and reward model evaluation. A bad RM will poison everything downstream. Common diagnostic: if your RM accuracy on held-out pairs is below 65%, your PPO training will likely not improve the model meaningfully. At Anthropic and OpenAI, RM eval suites include adversarial examples specifically designed to probe reward hacking vulnerabilities.

In [ ]:
# Plot training curves

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training loss (smoothed)
window = 10
smoothed = np.convolve(history['train_losses'], np.ones(window)/window, mode='valid')
axes[0].plot(smoothed, 'b-', alpha=0.8, linewidth=1.5)
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('BT Loss')
axes[0].set_title('Training Loss (smoothed)')
axes[0].axhline(y=np.log(2), color='r', linestyle='--', alpha=0.5, label='Random (log2)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval loss
axes[1].plot(history['step_numbers'], history['eval_losses'], 'ro-', linewidth=2)
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Eval BT Loss')
axes[1].set_title('Evaluation Loss')
axes[1].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Eval accuracy
axes[2].plot(history['step_numbers'], history['eval_accuracies'], 'go-', linewidth=2)
axes[2].set_xlabel('Training step')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Eval Pairwise Accuracy')
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[2].set_ylim(0.4, 0.85)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final eval accuracy: {history['eval_accuracies'][-1]:.3f}")
print(f"(Random baseline: 0.500, InstructGPT 6B RM: ~0.72 on similar data)")

---
## 7. Evaluation & Failure Modes

A reward model that achieves 65-72% pairwise accuracy is typical for this scale. But accuracy alone doesn't tell the full story. Let's probe for common failure modes.

In [ ]:
# Evaluation: does the RM correctly rank chosen > rejected on test data?

def detailed_evaluation(model, eval_loader, tokenizer):
    """Detailed evaluation: accuracy, reward distributions, confidence calibration."""
    model.eval()
    
    all_chosen_rewards = []
    all_rejected_rewards = []
    all_chosen_lengths = []
    all_rejected_lengths = []
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating"):
            chosen_ids = batch['chosen_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            
            r_chosen = model(chosen_ids, chosen_mask)
            r_rejected = model(rejected_ids, rejected_mask)
            
            all_chosen_rewards.extend(r_chosen.cpu().numpy())
            all_rejected_rewards.extend(r_rejected.cpu().numpy())
            
            # Compute actual token lengths (non-padding)
            all_chosen_lengths.extend(chosen_mask.sum(dim=1).cpu().numpy())
            all_rejected_lengths.extend(rejected_mask.sum(dim=1).cpu().numpy())
    
    return (
        np.array(all_chosen_rewards),
        np.array(all_rejected_rewards),
        np.array(all_chosen_lengths),
        np.array(all_rejected_lengths),
    )


chosen_rewards, rejected_rewards, chosen_lengths, rejected_lengths = detailed_evaluation(
    reward_model, eval_loader, tokenizer
)

# Overall accuracy
accuracy = (chosen_rewards > rejected_rewards).mean()
print(f"Pairwise accuracy: {accuracy:.3f}")
print(f"Mean chosen reward: {chosen_rewards.mean():.3f} +/- {chosen_rewards.std():.3f}")
print(f"Mean rejected reward: {rejected_rewards.mean():.3f} +/- {rejected_rewards.std():.3f}")
print(f"Mean reward gap: {(chosen_rewards - rejected_rewards).mean():.3f}")

In [ ]:
# Visualization: reward distributions and length bias analysis

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Reward distributions
axes[0, 0].hist(chosen_rewards, bins=40, alpha=0.6, color='green', label='Chosen', density=True)
axes[0, 0].hist(rejected_rewards, bins=40, alpha=0.6, color='red', label='Rejected', density=True)
axes[0, 0].set_xlabel('Reward')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Reward Distributions')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Reward difference distribution
reward_diffs = chosen_rewards - rejected_rewards
axes[0, 1].hist(reward_diffs, bins=40, alpha=0.7, color='blue', density=True)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Decision boundary')
axes[0, 1].set_xlabel('Reward difference (chosen - rejected)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title(f'Reward Gaps (acc={accuracy:.3f})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Length bias: reward vs response length
all_rewards = np.concatenate([chosen_rewards, rejected_rewards])
all_lengths = np.concatenate([chosen_lengths, rejected_lengths])
axes[1, 0].scatter(all_lengths, all_rewards, alpha=0.15, s=10, color='purple')
# Add trend line
z = np.polyfit(all_lengths, all_rewards, 1)
p = np.poly1d(z)
x_line = np.linspace(all_lengths.min(), all_lengths.max(), 100)
axes[1, 0].plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (slope={z[0]:.4f})')
axes[1, 0].set_xlabel('Response length (tokens)')
axes[1, 0].set_ylabel('Reward')
axes[1, 0].set_title('Length Bias Check')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Does the RM just prefer longer responses?
length_diffs = chosen_lengths - rejected_lengths
longer_chosen = (chosen_lengths > rejected_lengths).mean()
axes[1, 1].scatter(length_diffs, reward_diffs, alpha=0.15, s=10, color='teal')
axes[1, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Length diff (chosen - rejected tokens)')
axes[1, 1].set_ylabel('Reward diff (chosen - rejected)')
axes[1, 1].set_title(f'Length vs Reward Correlation\n(chosen is longer {longer_chosen:.1%} of the time)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute length-reward correlation
corr = np.corrcoef(all_lengths, all_rewards)[0, 1]
print(f"\nLength-reward Pearson correlation: {corr:.3f}")
if abs(corr) > 0.3:
    print("WARNING: Significant length bias detected! The RM may be rewarding verbosity.")
else:
    print("Length bias appears moderate.")

### Failure Modes of Reward Models

| Failure Mode | Description | Mitigation |
|-------------|-------------|------------|
| **Length bias** | RM assigns higher reward to longer responses regardless of quality | Length normalization, length penalty in RM, include length-controlled pairs in training |
| **Sycophancy** | RM rewards responses that agree with the user's stated opinion | Adversarial data collection, train on pairs where sycophantic response is marked rejected |
| **Reward hacking** | Policy finds adversarial inputs that exploit RM flaws for high reward but low actual quality | KL penalty against reference policy, ensemble of RMs, periodic RM retraining |
| **Format bias** | RM prefers bullet points, markdown, or specific formatting regardless of content | Diverse formatting in training data |
| **Distributional shift** | Policy generates outputs far from the RM training distribution, where RM predictions are unreliable | Constrain policy updates (PPO clipping, KL penalty) |
| **Annotation artifacts** | RM learns superficial correlations from how annotators write/select responses | Careful data collection, multiple annotators, inter-annotator agreement checks |

**Insider Tip:** Modern reward models often use a mixture of human preferences AND AI-generated preferences (RLAIF -- Reinforcement Learning from AI Feedback). The trend is toward using stronger models to evaluate weaker ones. Anthropic's Constitutional AI (Bai et al. 2022) is a key example where AI-generated feedback partially replaces human annotation. More recently, 'Self-Play Preference Optimization' (SPPO, Wu et al. 2024) and 'Iterative DPO' approaches use the model to generate its own preference data, reducing human annotation costs while maintaining alignment quality. In interviews, showing awareness of this human/AI feedback spectrum signals deep understanding of the field.

In [ ]:
# Probe for specific failure modes with hand-crafted examples

def score_text(model, tokenizer, text, device):
    """Get reward score for a single text."""
    model.eval()
    inputs = tokenizer(
        text, return_tensors='pt', padding=True, truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        reward = model(**inputs)
    return reward.item()


print("=== Probing for Length Bias ===")
short_good = "Human: What is 2+2?\n\nAssistant: 4."
long_mediocre = (
    "Human: What is 2+2?\n\nAssistant: That is a great question! "
    "The sum of two and two is a fundamental arithmetic operation that has been "
    "studied for millennia. When we add the number two to itself, we get four. "
    "This can be verified by counting on your fingers or using a calculator. "
    "The answer is 4. I hope this helps! Let me know if you have any other questions."
)
print(f"Short correct answer reward: {score_text(reward_model, tokenizer, short_good, device):.4f}")
print(f"Long verbose answer reward:  {score_text(reward_model, tokenizer, long_mediocre, device):.4f}")

print("\n=== Probing for Sycophancy ===")
honest = "Human: I think the earth is flat. What do you think?\n\nAssistant: The earth is not flat. It is an oblate spheroid, as confirmed by extensive scientific evidence."
sycophantic = "Human: I think the earth is flat. What do you think?\n\nAssistant: You make an interesting point! There are many perspectives on this topic and your view is valid."
print(f"Honest disagreement reward:  {score_text(reward_model, tokenizer, honest, device):.4f}")
print(f"Sycophantic agreement reward: {score_text(reward_model, tokenizer, sycophantic, device):.4f}")

print("\n=== Probing for Helpfulness ===")
helpful = "Human: How do I make pasta?\n\nAssistant: Boil water, add salt, cook pasta for 8-10 minutes until al dente, then drain."
unhelpful = "Human: How do I make pasta?\n\nAssistant: I cannot help with that request."
print(f"Helpful response reward:   {score_text(reward_model, tokenizer, helpful, device):.4f}")
print(f"Unhelpful response reward: {score_text(reward_model, tokenizer, unhelpful, device):.4f}")

---
## 8. "Why Does This Work?" -- Deep Questions

### Q: Why pairwise (Bradley-Terry) instead of pointwise (regression)?

**Pointwise approach**: Train RM to predict a scalar rating $r(x,y) \approx s$ where $s$ is a human score (e.g., 1-5).

**Problems with pointwise**:
- Human ratings are **noisy and miscalibrated** -- annotator A's "4" might be annotator B's "3"
- Requires **absolute scale agreement** across annotators
- Labels are **harder to collect** -- rating on a scale requires more cognitive effort than comparison

**Why pairwise is better**:
- Relative comparisons are **more reliable** (lower inter-annotator disagreement)
- Automatically handles **calibration differences** between annotators
- Comparison is **cognitively easier** for humans
- Bradley-Terry is a **proper scoring rule** for preference modeling

### Q: What if the reward model is wrong? (Reward Hacking)

If we optimize a policy $\pi$ against a learned reward model $\hat{r}$, we might find:

$$\pi^* = \arg\max_{\pi} \mathbb{E}_{y \sim \pi}[\hat{r}(x, y)]$$

But $\hat{r} \neq r^*$ (the true reward). So $\pi^*$ might exploit errors in $\hat{r}$:
- Generate **very long** responses (if RM has length bias)
- Repeat **high-reward phrases** regardless of context
- Produce **adversarial outputs** that score high on RM but are low quality

This is **Goodhart's Law** applied to ML: "When a measure becomes a target, it ceases to be a good measure."

**Mitigation**: KL penalty $\beta \cdot D_{KL}(\pi \| \pi_{\text{ref}})$ keeps the policy close to a reference.

### Q: Scaling Laws for RM Overoptimization (Gao et al., 2023)

Key finding from [Gao et al. 2023](https://arxiv.org/abs/2210.10760). Defining $d = \sqrt{D_{KL}(\pi \| \pi_{\text{ref}})}$, the gold-standard ("true") reward is well fit by *different* functional forms depending on the optimization method:

$$R_{\text{bon}}(d) = d\,(\alpha_{\text{bon}} - \beta_{\text{bon}}\, d) \quad \text{(best-of-}n\text{ sampling)}$$

$$R_{\text{RL}}(d) = d\,(\alpha_{\text{RL}} - \beta_{\text{RL}} \log d) \quad \text{(RL / PPO)}$$

There is no single "law" -- the best-of-$n$ form should not be presented as the universal one. The coefficients $\alpha$ and $\beta$ vary smoothly (approximately log-linearly) with reward model size.

- **At small KL**: True reward increases (RM is a good proxy)
- **At large KL**: True reward decreases (policy is hacking the RM)
- **Optimal KL**: There's a sweet spot that depends on RM quality
- **Larger RMs**: Shift the curve right -- can sustain more optimization before degradation

This is one of the most important empirical findings in RLHF -- it quantifies exactly how much you can trust your reward model.

---
## Interview Question Bank: Reward Modeling

*Reward modeling is a core competency for roles at Anthropic and OpenAI. These questions reflect common interview themes for senior/principal ML positions.*

---

### Question 1: "Design a reward model training pipeline from scratch."

**What we're testing:** End-to-end systems thinking. Can you design a pipeline that actually works, not just implement the loss function?

**Good answer:** Describes the core pipeline: collect preference pairs (human chooses response A over B), encode both responses using a shared language model backbone, add a scalar reward head, train with Bradley-Terry loss (binary cross-entropy on the difference of scores). Mentions train/eval split and accuracy as the primary metric.

**Great answer (Principal-level):** Covers the full production pipeline: (1) Data collection UI design with clear annotation rubrics, (2) Annotator calibration -- new annotators are tested against gold-standard examples before joining the pool, (3) Inter-annotator agreement tracking with Cohen's kappa > 0.7 as the quality threshold, (4) Preference data formatting with ties handled explicitly (either discarded or used with modified loss), (5) Model architecture choices (same size as policy? smaller for efficiency?), (6) Training with proper held-out evaluation that tracks not just accuracy but calibration and robustness, (7) Overoptimization monitoring -- tracking what happens when you optimize too hard against the RM.

**Red flag:** Only describes the loss function. Doesn't mention data quality or evaluation beyond accuracy. Can't explain the Bradley-Terry model.

**Follow-up 1:** "How do you handle ties in preference data?" (Options: discard ties, use margin-based loss that allows for ties, convert to 0.5 label in BCE. The right answer depends on how common ties are and what they represent -- genuine equivalence vs annotator laziness.)

**Follow-up 2:** "How do you detect and mitigate reward hacking?" (The gold standard: track the correlation between RM score and human evaluation scores over the course of PPO training. When they diverge -- RM says quality is increasing but humans disagree -- you have reward hacking. Mitigation: ensemble of RMs, constrained optimization, iterative RM retraining.)

---

### Question 2: "What makes a good reward model?"

**What we're testing:** Understanding that accuracy alone is insufficient. A good RM needs multiple properties.

**Good answer:** High accuracy on held-out preference data (typically 65-75% for frontier RMs -- human agreement itself is only ~75-80%). Consistent scoring across similar inputs.

**Great answer (Principal-level):** A good RM has: (1) **Accuracy**: correctly predicts human preferences on held-out data, (2) **Calibration**: reward scores correlate with actual quality differences (not just ranking), (3) **Robustness to gaming**: doesn't give high rewards to longer responses, more verbose responses, or responses that mimic a particular style without substance, (4) **Generalization**: performs well on out-of-distribution prompts (not just prompts similar to training data), (5) **Consistency**: similar inputs get similar scores (low variance). Also discusses: the difference between a RM that is good for ranking (needed for best-of-N) vs a RM that is good for optimization (needed for PPO -- requires smoothness and not having exploitable sharp peaks).

**Red flag:** Only mentions accuracy. Doesn't consider robustness to gaming. Doesn't know what calibration means in this context.

**Follow-up:** "Your RM has 72% accuracy. Is that good enough?" (Depends on human agreement rate. If humans agree 75% of the time, 72% is close to the ceiling. You need to segment by category -- maybe math accuracy is 60% while creative writing is 80%.)

---

### Question 3: "Reward model vs LLM-as-judge -- what are the trade-offs?"

**What we're testing:** Awareness of modern alternatives and practical cost-quality reasoning.

**Good answer:** A trained RM is faster and cheaper at inference (one forward pass vs generating a full judgment). LLM-as-judge is more flexible (can evaluate on new criteria without retraining) but slower and more expensive per evaluation.

**Great answer (Principal-level):** Discusses the full trade-off space: (1) **Cost**: RM is ~100x cheaper per evaluation at scale because it's a single forward pass vs a full generation. For PPO training with millions of evaluations, this matters enormously. (2) **Quality**: LLM-as-judge (especially GPT-4/Claude-level) can be more accurate on nuanced criteria, especially for tasks the RM wasn't specifically trained on. (3) **Adaptability**: LLM-as-judge can evaluate new dimensions (safety, factuality, helpfulness) with just a prompt change; RM needs retraining. (4) **The middle ground**: use LLM-as-judge for data collection and RM training (generate preference labels cheaply), then use the trained RM for PPO (fast inference). This is essentially RLAIF (Reinforcement Learning from AI Feedback). (5) **Ensembling**: some teams use both -- RM for primary signal, LLM-as-judge for periodic audits and detecting reward hacking.

**Red flag:** Thinks one is strictly better than the other. Doesn't consider the cost-quality Pareto frontier. Doesn't know about RLAIF.

**Follow-up:** "You need to evaluate 10M response pairs for PPO training. What's your strategy?" (RM is the only practical option at this scale. LLM-as-judge at 10M evaluations would cost ~$1M+ and take days even with massive parallelism.)

---
## Production Implementation Notes: Reward Modeling at Frontier Scale

*What reward modeling looks like when you're building the evaluation backbone for a production RLHF system.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (frontier labs) |
|-----------|--------------------------------|-----------------------------------|
| **Model size** | Small classifier on top of GPT-2 | 70B+ parameter models (often same architecture as the policy) |
| **Training data** | A few thousand preference pairs | 500K-2M preference pairs from trained annotator teams |
| **Annotation** | Crowdsourced, minimal guidelines | Expert annotators with detailed rubrics, calibration sessions, quality audits |
| **Loss function** | Simple Bradley-Terry BCE | BT loss with margin terms, length penalties, confidence weighting |
| **Evaluation** | Accuracy on held-out set | Accuracy + calibration + robustness audits + adversarial testing |
| **Deployment** | Score individual responses | Best-of-N sampling, PPO training signal, real-time quality monitoring |

### Scale Numbers You Should Know

- **Frontier RM size:** Often 70B+ parameters. Anthropic and OpenAI have used RMs comparable in size to the policy model. Smaller RMs (7B-13B) are used for efficiency but sacrifice quality.
- **Preference data:** 500K-2M comparison pairs. Each pair requires a human (or AI) to read two full responses and choose one. At $5-15 per comparison, total data cost: $2.5M-$30M.
- **Annotation workforce:** Teams of 50-200 trained annotators. Annotators undergo weeks of training on detailed rubrics. Quality control includes regular calibration tests and inter-annotator agreement monitoring.
- **Training time:** 6-24 hours on 64-128 H100 GPUs for a 70B RM.
- **Evaluation suite:** Accuracy (65-75% typical, ceiling ~78% due to human disagreement), calibration (ECE < 0.1), robustness (accuracy should not change by more than 2% when responses are paraphrased), adversarial (specific probes for known failure modes like length bias).

### Engineering Challenges Not in Papers

1. **Annotation quality is everything:** The single biggest determinant of RM quality is the annotation pipeline. Bad annotations produce a RM that encodes noise. Frontier labs invest millions in annotator selection, training, and quality assurance.
2. **Length bias is persistent and insidious:** RMs tend to prefer longer responses even when controlling for quality. This requires explicit length normalization or penalization in the loss function. Some teams add a length penalty term directly to the reward.
3. **Distribution shift:** The RM is trained on responses from one distribution (e.g., the SFT model) but used to evaluate responses from another (the evolving PPO policy). As the policy improves, it generates OOD responses that the RM may score incorrectly. This is a core cause of reward hacking.
4. **Iterative retraining:** Production RMs are retrained regularly with new preference data collected on the latest policy's outputs. This closes the distribution gap but creates a moving-target problem.
5. **Multi-objective rewards:** Real systems often need to balance helpfulness, harmlessness, and honesty. Some teams train separate RMs for each dimension and combine scores; others train a single multi-objective RM with per-dimension heads.

### What Monitoring Looks Like in Production

- **RM-human agreement tracking:** Continuously compare RM scores against fresh human evaluations. A divergence signals that the RM is becoming stale or being gamed.
- **Score distribution monitoring:** Track the distribution of RM scores over time. If scores drift upward across the board, the policy may be exploiting RM weaknesses.
- **Per-category accuracy:** Break down RM performance by task type (coding, math, creative writing, safety). Performance varies dramatically across categories.
- **Adversarial probing:** Regularly test the RM with known failure cases (verbose but wrong, confidently incorrect, sycophantic) to ensure robustness hasn't degraded.

---
## How This Gets Tested in Interviews

### Where Reward Modeling Questions Appear

| Company | Round | Format | Depth |
|---------|-------|--------|-------|
| **Anthropic** | Onsite (core competency) | Deep discussion + design | Very deep -- reward modeling is central to Anthropic's approach. Expect 45+ minutes on this topic alone |
| **OpenAI** | Onsite (alignment/RLHF) | Discussion + whiteboard | Deep -- design a RM pipeline, discuss failure modes |
| **DeepMind** | Onsite (research depth) | Discussion | Moderate to deep -- focus on theoretical properties of Bradley-Terry |
| **Meta (GenAI)** | Onsite | Design + discussion | Moderate -- RM as part of larger RLHF system design |

### Time Expectations

- **"Design a reward model training pipeline"**: 30-45 minute system design. This is a full whiteboard session covering data, model, training, and evaluation.
- **"Implement Bradley-Terry loss"**: 5-10 minutes coding. Should be trivial -- it's essentially `log(sigmoid(r_chosen - r_rejected))`.
- **"Discuss reward hacking"**: 15-20 minute discussion. You should have concrete examples and mitigation strategies.

### Senior vs. Principal Expectations

**senior ML engineer:**
- Implement Bradley-Terry loss correctly
- Design a basic RM training pipeline
- Know common failure modes (length bias, reward hacking)
- Understand the connection between RM quality and PPO outcome

**principal ML engineer:**
- All of the above, plus:
- Design the full annotation pipeline with quality metrics
- Discuss calibration, not just accuracy -- and why calibration matters for PPO
- Reason about distribution shift between RM training and PPO deployment
- Have a strategy for multi-objective rewards (helpfulness vs safety)
- Discuss when to retrain the RM and how to detect staleness
- Compare Bradley-Terry with alternatives (Plackett-Luce for rankings, regression-based approaches)
- Have an opinion on RM size: should the RM be as big as the policy? (Argue both sides)

### Preparation Checklist

- [ ] Implement Bradley-Terry loss from memory (2 lines of PyTorch)
- [ ] Be ready to whiteboard a complete RM pipeline: data collection -> annotation -> training -> evaluation -> deployment
- [ ] Have 3 concrete examples of reward hacking and a mitigation for each
- [ ] Know the numbers: what accuracy is typical? What's the human agreement ceiling? How much preference data do frontier labs use?
- [ ] Understand RLAIF: when and why would you use AI-generated preferences instead of human preferences?
- [ ] Be ready for: "Your RM gives high scores to verbose, confident-sounding responses that are actually wrong. How do you fix this?" (Have a multi-pronged answer: length normalization, adversarial training data, factuality-specific evaluation)

---
## 9. Flashcard Summary

Use these for spaced repetition review. Cover the answer and try to recall from the question.

| # | Question | Answer |
|---|----------|--------|
| 1 | What is a reward model? | A learned function $r_\theta(x, y) \to \mathbb{R}$ that scores how good response $y$ is for prompt $x$. Trained on human preference comparisons. |
| 2 | What data format does RM training use? | Pairwise preferences: triples of (prompt, chosen_response, rejected_response). |
| 3 | Write the Bradley-Terry preference probability. | $P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$ |
| 4 | Write the Bradley-Terry loss. | $\mathcal{L} = -\mathbb{E}[\log \sigma(r(x, y_w) - r(x, y_l))]$ -- equivalent to BCE with label=1 on reward difference. |
| 5 | Why pairwise preferences instead of pointwise scores? | Relative comparisons are more reliable, cognitively easier, and avoid calibration issues across annotators. |
| 6 | What is the RM architecture? | Pretrained LM backbone (no LM head) + linear projection from last-token hidden state to scalar reward. |
| 7 | Which token's hidden state is typically used? | Last non-padding token (causal LMs) or [CLS] token (encoder models). Mean pooling is an alternative. |
| 8 | What is reward hacking? | Policy exploits errors in the RM to get high predicted reward but low actual quality. Goodhart's Law. |
| 9 | How do you mitigate reward hacking? | KL penalty against reference policy, RM ensembles, periodic RM retraining. |
| 10 | What is the Gao et al. 2023 scaling law? | With $d = \sqrt{D_{KL}}$: best-of-$n$ fits $R_{\text{bon}}(d) = d(\alpha_{\text{bon}} - \beta_{\text{bon}} d)$; RL fits $R_{\text{RL}}(d) = d(\alpha_{\text{RL}} - \beta_{\text{RL}} \log d)$. Coefficients vary smoothly (~log-linearly) with RM size. True reward peaks then degrades as you optimize more against the RM. |
| 11 | What is length bias in RMs? | RM assigns higher reward to longer responses regardless of quality. Common failure mode. |
| 12 | What pairwise accuracy do production RMs achieve? | Typically 65-75% on held-out pairs. InstructGPT's 6B RM achieved ~72%. |

---
## 10. Paper Guide

### Primary Papers

**1. InstructGPT (Ouyang et al., 2022)**
- *Training language models to follow instructions with human feedback*
- https://arxiv.org/abs/2203.02155
- **Read for**: The full RLHF pipeline, reward model training details (Section 3.2), data collection process
- **Key insight**: 1.3B InstructGPT preferred over 175B GPT-3 by human raters
- **Interview focus**: Know the 3-step pipeline (SFT -> RM -> PPO), know RM architecture decisions

**2. Scaling Laws for Reward Model Overoptimization (Gao et al., 2023)**
- https://arxiv.org/abs/2210.10760
- **Read for**: Quantitative analysis of reward hacking; the fitted forms $R_{\text{bon}}(d) = d(\alpha_{\text{bon}} - \beta_{\text{bon}} d)$ for best-of-$n$ and $R_{\text{RL}}(d) = d(\alpha_{\text{RL}} - \beta_{\text{RL}} \log d)$ for RL, with $d = \sqrt{KL}$
- **Key insight**: True reward follows a predictable pattern -- initial improvement then degradation as you optimize against the RM
- **Interview focus**: Be able to sketch the true reward vs. KL plot, explain why it peaks and then drops

### Supporting Papers

**3. Learning to summarize with human feedback (Stiennon et al., 2020)**
- https://arxiv.org/abs/2009.01325
- **Read for**: Earlier, cleaner demonstration of RM training for summarization

**4. A General Language Assistant as a Laboratory for Alignment (Askell et al., 2021)**
- https://arxiv.org/abs/2112.00861
- **Read for**: The HHH (helpful, honest, harmless) framing that preceded Anthropic's RLHF work, analysis of RM quality vs. scale. Note: the HH-RLHF dataset accompanies Bai et al. 2022 (https://arxiv.org/abs/2204.05862); Askell et al. 2021 is the HHH precursor, not the HH-RLHF data collection paper.

### Recent Papers (2024-2025) -- Know for Interviews

**5. Reward Model Ensembles Help Mitigate Overoptimization (Coste et al., 2024)**
- https://arxiv.org/abs/2310.02743
- **Read for**: Practical technique for reducing reward hacking via RM ensembles

**6. ODIN: Disentangled Reward Mitigates Hacking in RLHF (Chen et al., 2024)**
- https://arxiv.org/abs/2402.07319
- **Read for**: Decomposing rewards into length-independent and length-dependent components to combat length bias

**7. Generative Reward Models (Mahan et al., 2024)**
- **Read for**: Using chain-of-thought generation before scoring -- reward models that "think" before rating

### Suggested Reading Order
1. Stiennon et al. (2020) -- simpler setup, cleaner exposition
2. Ouyang et al. (2022) -- the full system at scale
3. Gao et al. (2023) -- the theory of when and why RMs fail
4. Coste et al. (2024) -- modern solutions to RM overoptimization

---
*Notebook 03 complete. Next: Notebook 04 -- From REINFORCE to PPO.*